# Token-Level Feature Engineering for NER

In this notebook, we explore how classical NLP systems represent tokens using manually designed features.

Before deep learning, models did not learn representations automatically. Instead, humans designed features that helped the model decide whether a token was part of an entity.

## Learning Objectives

By the end of this notebook, you should be able to:
- explain what token-level features are
- create lexical, orthographic, syntactic, and contextual features
- understand why feature engineering was central in classical NLP
- inspect how features can help with Named Entity Recognition (NER)

---

## 1. Motivation

Named Entity Recognition is the task of identifying entities such as persons, organizations, locations, dates, or monetary values in text.

Example:

```text
Google acquired DeepMind in London in 2014.
```

A NER system should recognize:

| Entity | Label |
|---|---|
| Google | ORG |
| DeepMind | ORG |
| London | GPE |
| 2014 | DATE |

Classical ML models cannot directly understand raw text. We first need to transform each token into a set of features.


In [ ]:
import spacy
import pandas as pd

## 2. Load spaCy

We use spaCy for tokenization and part-of-speech tagging.

If the model is not installed, run:

```bash
python -m spacy download en_core_web_sm
```

In [ ]:
nlp = spacy.load("en_core_web_sm")

## 3. Example Text

We start with a short sentence containing different types of named entities.

In [ ]:
text = "Google acquired DeepMind in London in 2014."
doc = nlp(text)

for token in doc:
    print(token.text)

## 4. Inspect Token Information

Each token contains useful information such as:

- token text
- part-of-speech tag
- whether it is alphabetic
- whether it is a digit
- whether it starts with an uppercase letter

These properties can become features for a classical ML model.

In [ ]:
rows = []

for token in doc:
    rows.append({
        "token": token.text,
        "pos": token.pos_,
        "is_alpha": token.is_alpha,
        "is_digit": token.is_digit,
        "is_title": token.text.istitle(),
        "is_upper": token.text.isupper(),
    })

pd.DataFrame(rows)

## 5. Types of Token Features

Classical NLP systems often used different feature types:

### Lexical Features
- token text
- lowercase token
- prefixes and suffixes
- token length

### Orthographic Features
- capitalization
- all uppercase
- contains digits
- punctuation

### Syntactic Features
- part-of-speech tag
- dependency relation

### Contextual Features
- previous token
- next token
- previous POS tag
- next POS tag


## 6. Build a Feature Function

A feature function converts a token into a dictionary of properties.

This dictionary can later be transformed into a numerical vector for a machine learning model.

In [ ]:
def basic_token_features(token):
    """Create simple features for one token."""
    return {
        "token.lower": token.text.lower(),
        "token.length": len(token.text),
        "token.is_title": token.text.istitle(),
        "token.is_upper": token.text.isupper(),
        "token.is_digit": token.text.isdigit(),
        "prefix_2": token.text[:2],
        "suffix_2": token.text[-2:],
        "pos": token.pos_,
    }


basic_token_features(doc[0])

## 7. Apply the Feature Function to All Tokens

Now we create a feature representation for each token in the sentence.

In [ ]:
feature_rows = []

for token in doc:
    features = basic_token_features(token)
    features["token"] = token.text
    feature_rows.append(features)

pd.DataFrame(feature_rows)

## 8. Add Context Features

For NER, context is extremely important.

Example:

```text
Apple released a new MacBook.
I ate an apple yesterday.
```

The token `Apple/apple` has different meanings depending on nearby words.

Classical NLP systems often used a fixed context window around the current token.

In [ ]:
def token_features(doc, i):
    """Create token-level features with local context."""
    token = doc[i]
    
    features = {
        # lexical features
        "token.lower": token.text.lower(),
        "token.length": len(token.text),
        "prefix_2": token.text[:2],
        "prefix_3": token.text[:3],
        "suffix_2": token.text[-2:],
        "suffix_3": token.text[-3:],
        
        # orthographic features
        "is_title": token.text.istitle(),
        "is_upper": token.text.isupper(),
        "is_digit": token.text.isdigit(),
        "is_alpha": token.is_alpha,
        
        # syntactic features
        "pos": token.pos_,
    }
    
    # previous token features
    if i > 0:
        prev_token = doc[i - 1]
        features.update({
            "prev.lower": prev_token.text.lower(),
            "prev.is_title": prev_token.text.istitle(),
            "prev.pos": prev_token.pos_,
        })
    else:
        features["BOS"] = True  # Beginning of sentence
    
    # next token features
    if i < len(doc) - 1:
        next_token = doc[i + 1]
        features.update({
            "next.lower": next_token.text.lower(),
            "next.is_title": next_token.text.istitle(),
            "next.pos": next_token.pos_,
        })
    else:
        features["EOS"] = True  # End of sentence
    
    return features

## 9. Inspect Context Features

Let's inspect the feature representation for each token.

In [ ]:
feature_rows = []

for i, token in enumerate(doc):
    features = token_features(doc, i)
    features["token"] = token.text
    feature_rows.append(features)

pd.DataFrame(feature_rows)

## 10. Why Context Matters

Now we compare two sentences where the same word has different meanings.

In [ ]:
sentences = [
    "Apple released a new MacBook.",
    "I ate an apple yesterday."
]

for sentence in sentences:
    doc_tmp = nlp(sentence)
    print("Sentence:", sentence)
    for i, token in enumerate(doc_tmp):
        if token.text.lower() == "apple":
            print(token_features(doc_tmp, i))
    print("-" * 80)

### Discussion

Look at the features for `Apple` in both sentences.

Questions:

1. Which features are the same?
2. Which features are different?
3. Which features could help a model distinguish between organization and fruit?


## 11. Gazetteer Features

A gazetteer is a curated list of known entities.

Example:

```python
["Google", "Apple", "Microsoft", "DeepMind"]
```

Gazetteers were commonly used in classical NER systems as lookup features.

In [ ]:
ORG_GAZETTEER = {"google", "apple", "microsoft", "deepmind", "amazon"}
GPE_GAZETTEER = {"london", "berlin", "paris", "hamburg"}


def token_features_with_gazetteer(doc, i):
    features = token_features(doc, i)
    token_lower = doc[i].text.lower()
    
    features["in_org_gazetteer"] = token_lower in ORG_GAZETTEER
    features["in_gpe_gazetteer"] = token_lower in GPE_GAZETTEER
    
    return features

In [ ]:
rows = []

for i, token in enumerate(doc):
    features = token_features_with_gazetteer(doc, i)
    features["token"] = token.text
    rows.append(features)

pd.DataFrame(rows)[["token", "in_org_gazetteer", "in_gpe_gazetteer", "pos", "is_title"]]

## 12. Token Features and NER Labels

In supervised learning, each token must have a label.

For example:

| Token | Label |
|---|---|
| Google | B-ORG |
| acquired | O |
| DeepMind | B-ORG |
| in | O |
| London | B-GPE |
| in | O |
| 2014 | B-DATE |
| . | O |

The next notebook will use features like the ones above to train a classical ML classifier.

In [ ]:
labels = ["B-ORG", "O", "B-ORG", "O", "B-GPE", "O", "B-DATE", "O"]

rows = []
for i, token in enumerate(doc):
    rows.append({
        "token": token.text,
        "label": labels[i],
        "features": token_features_with_gazetteer(doc, i)
    })

pd.DataFrame([{"token": r["token"], "label": r["label"]} for r in rows])

## 13. Mini Task

Now create features for the following sentence:

```text
Amazon opened an office in Paris.
```

Tasks:

1. Tokenize the sentence.
2. Create token-level features.
3. Add gazetteer features.
4. Inspect which features could help identify entities.


In [ ]:
new_text = "Amazon opened an office in Paris."
new_doc = nlp(new_text)

# TODO: create a feature table for the new sentence
rows = []

for i, token in enumerate(new_doc):
    features = token_features_with_gazetteer(new_doc, i)
    features["token"] = token.text
    rows.append(features)

pd.DataFrame(rows)

## 14. Reflection

Answer briefly:

1. Which features seem useful for detecting organizations?
2. Which features seem useful for detecting locations?
3. Why are context features useful?
4. What are the limitations of gazetteers?
5. Why does classical NLP require so much manual feature engineering?


## Summary

In this notebook, we explored token-level feature engineering for NER.

We created:

- lexical features
- orthographic features
- syntactic features
- contextual features
- gazetteer features

Main takeaway:

> Classical NLP systems rely on humans manually designing useful representations.

In the next notebook, we will use these features to train a classical ML classifier.